# YOLOv8 Floor Plan Door/Window Detection — Training Notebook

Train a YOLOv8 model to detect **doors** and **windows** in architectural floor plan images.

**Platforms**: Google Colab (free GPU) or AWS SageMaker

**Google Colab:** use `train_yolo_colab.ipynb` and [COLAB.md](COLAB.md) for a ready-made flow (GPU check, Drive, zip upload, download).

## Prerequisites
- Annotated dataset in YOLO format (use `ml/dataset.py` locally to prepare the folder structure)
- Annotations created with LabelImg, CVAT, or Roboflow
- Classes: `0 = door`, `1 = window`

## 1. Setup Environment

In [ ]:
# Install ultralytics (includes YOLOv8 + PyTorch)
!pip install -q ultralytics

import ultralytics
ultralytics.checks()

# For Google Colab: mount Drive to access dataset
# from google.colab import drive
# drive.mount('/content/drive')

## 2. Upload / Link Dataset

Upload your YOLO-format dataset or point to it on Google Drive / S3.

Expected structure:
```
dataset/
  data.yaml
  images/
    train/  (*.png)
    val/    (*.png)
  labels/
    train/  (*.txt — YOLO format: class cx cy w h)
    val/    (*.txt)
```

In [ ]:
import os
from pathlib import Path

# --- Option A: Upload a zip of your dataset ---
# from google.colab import files
# uploaded = files.upload()  # upload dataset.zip
# !unzip -q dataset.zip -d /content/dataset

# --- Option B: Link from Google Drive ---
# DATASET_ROOT = Path('/content/drive/MyDrive/Plan2BoQ/dataset')

# --- Option C: Local path (AWS / local GPU) ---
DATASET_ROOT = Path('dataset')

DATA_YAML = DATASET_ROOT / 'data.yaml'
assert DATA_YAML.exists(), f'data.yaml not found at {DATA_YAML}'

# Print dataset stats
train_imgs = list((DATASET_ROOT / 'images' / 'train').glob('*.png'))
val_imgs = list((DATASET_ROOT / 'images' / 'val').glob('*.png'))
print(f'Training images: {len(train_imgs)}')
print(f'Validation images: {len(val_imgs)}')

## 3. Train YOLOv8

Using `yolov8s.pt` (small) as the base — good balance of speed and accuracy for floor plans.

Key hyperparameters:
- `imgsz=1280` — floor plans have fine details, larger input helps
- `epochs=100` with `patience=20` (early stopping)
- `batch=8` — adjust based on GPU memory (Colab free: 8-16, A100: 32+)

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8s (transfer learning from COCO)
model = YOLO('yolov8s.pt')

# Train
results = model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=1280,
    batch=8,
    patience=20,
    project='runs/floorplan',
    name='door_window_v1',
    # Augmentation (built-in)
    hsv_h=0.0,       # no hue shift (floor plans are B&W)
    hsv_s=0.0,       # no saturation shift
    hsv_v=0.2,       # slight brightness variation
    degrees=5.0,     # slight rotation (plans can be slightly tilted)
    scale=0.3,       # scale variation
    flipud=0.0,      # no vertical flip (plans have orientation)
    fliplr=0.5,      # horizontal flip is OK
    mosaic=0.5,       # mosaic augmentation
)

## 4. Validation Metrics

In [ ]:
# Validate on the validation set
metrics = model.val()

print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

# Per-class breakdown
for i, name in enumerate(model.names.values()):
    print(f"  {name}: AP50={metrics.box.ap50[i]:.4f}  AP50-95={metrics.box.ap[i]:.4f}")

In [ ]:
# Visualise training curves
from IPython.display import Image, display
import glob

run_dir = Path('runs/floorplan/door_window_v1')

for plot in ['results.png', 'confusion_matrix.png', 'F1_curve.png', 'PR_curve.png']:
    p = run_dir / plot
    if p.exists():
        print(f'\n--- {plot} ---')
        display(Image(filename=str(p), width=800))

## 5. Export Best Model

Download the best checkpoint to use locally in the Plan2BoQ pipeline.

Copy `best.pt` to your local repo at `ml/best.pt` — the pipeline will auto-detect it.

In [ ]:
import shutil

best_model = run_dir / 'weights' / 'best.pt'
assert best_model.exists(), f'best.pt not found at {best_model}'

# Copy to a convenient location
export_path = Path('best.pt')
shutil.copy2(str(best_model), str(export_path))
print(f'Best model exported to: {export_path}')
print(f'File size: {export_path.stat().st_size / 1024 / 1024:.1f} MB')

# For Google Colab: download the file
# from google.colab import files
# files.download(str(export_path))

# For AWS: copy to S3
# !aws s3 cp best.pt s3://your-bucket/models/best.pt

## 6. Test Inference on Sample Images

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display

# Load the trained model
model = YOLO(str(export_path))

# Run on validation images
val_images = sorted((DATASET_ROOT / 'images' / 'val').glob('*.png'))[:5]

for img_path in val_images:
    results = model.predict(source=str(img_path), conf=0.25, save=True,
                            project='runs/floorplan', name='test_predictions')
    
    for r in results:
        boxes = r.boxes
        door_count = sum(1 for b in boxes if model.names[int(b.cls)] == 'door')
        window_count = sum(1 for b in boxes if model.names[int(b.cls)] == 'window')
        print(f'{img_path.name}: {door_count} doors, {window_count} windows')

# Display annotated results
pred_dir = Path('runs/floorplan/test_predictions')
for img in sorted(pred_dir.glob('*.png'))[:5]:
    print(f'\n--- {img.name} ---')
    display(Image(filename=str(img), width=800))

## Next Steps

1. Copy `best.pt` to your local repo at `Plan2BoQ - PDF cleaning/ml/best.pt`
2. Run `python3 process_cad.py` or `python3 process_floor_plans.py` — ML detection will activate automatically
3. Check `cleaned/*_ml_detect.json` and `cleaned/*_ml_annotated.png` for results

### Improving Accuracy
- Add more annotated images (target 200+ for decent results, 500+ for production)
- Review false positives/negatives in `confusion_matrix.png`
- Try `yolov8m.pt` (medium) if you have more GPU memory
- Fine-tune augmentation params if floor plans have consistent style